# Stage 5 — Source audio files for Model B

CLAUDE.md Stage 5 says to *de-risk audio sourcing early*:
confirm we can get audio for ~50 tracks before committing to the Stage 6 feature
pipeline. This notebook picks a small, **distribution-matched** subset of the
dataset and downloads each track's audio so we can later extract richer
`librosa` features and test whether they beat the 10 Spotify features
(current Model B: Spearman 0.198).

Downloads go through [`spotify_dl`](https://github.com/SathyaBhat/spotify-dl),
which reads track metadata from the Spotify API and fetches the matching audio
via `yt-dlp`. This is for **personal, educational feature extraction only** — not
redistribution. You are responsible for complying with the relevant terms of
service.

**Why a distribution-matched sample (not just the first 50 rows):** we want the
50 downloaded tracks to *look like* the whole dataset in audio-feature space, so
any conclusion we draw about librosa features generalizes. We match the
**marginal distribution** of each audio feature (see the sampling section).

## 0. Prerequisites (run once, in your shell — not here)

1. **Install the downloader and audio tooling**
   ```bash
   pip install spotify_dl
   brew install ffmpeg      # macOS; spotify_dl/yt-dlp needs ffmpeg to make mp3s
   ```
2. **Spotify API credentials.** `spotify_dl` needs a (free) Spotify app's
   client id/secret. Create one at
   https://developer.spotify.com/dashboard and export the credentials in the
   **same shell you launch Jupyter from**, so the notebook can read them from
   the environment:
   ```bash
   export SPOTIPY_CLIENT_ID=your_client_id
   export SPOTIPY_CLIENT_SECRET=your_client_secret
   ```
   > Security: never paste the secret into this notebook or commit it. The
   > notebook only *reads* it from the environment.

In [ ]:
# One-time install (uncomment to run inside the notebook kernel instead of the shell):
# %pip install spotify_dl scipy scikit-learn pandas pyarrow matplotlib
# ffmpeg must be installed system-wide (e.g. `brew install ffmpeg` on macOS).

In [ ]:
import os
import glob
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt

## 1. Config

In [ ]:
# Locate the repo root (the folder that contains `data/`), so this works
# regardless of where Jupyter was launched from.
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data").exists(), "Could not find the data/ folder from the current directory."

DATA_PATH = ROOT / "data" / "processed" / "orig_data_with_listeners.parquet"
AUDIO_DIR = ROOT / "data" / "audio"        # mp3s + manifest land here
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "spotify_track_id"                 # the real Spotify id (track_id is an int index)

# Audio-feature marginals we want the sample to match (what Stage 6 will re-estimate).
FEATURES = ["danceability", "energy", "loudness", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo", "duration_ms"]

N_SAMPLES = 50          # ~50 tracks is the Stage 5 de-risk target
N_DRAWS   = 2000        # candidate random subsets to search over (see sampling section)
SEED      = 667

DOWNLOAD_TIMEOUT = 240  # seconds per track before we give up and move on
SLEEP_BETWEEN    = 1.0  # politeness pause between downloads (seconds)

## 2. Load the dataset

In [ ]:
df = pd.read_parquet(DATA_PATH)

# The population we sample from: rows with all features present and a usable
# Spotify id, deduplicated by id (a track can appear under several genres).
pop = (df.dropna(subset=FEATURES + [ID_COL])
         .drop_duplicates(ID_COL)
         .reset_index(drop=True))
print(f"Usable population: {len(pop):,} tracks")
pop[[ID_COL, "track_name", "artists"] + FEATURES].head()

## 3. Distribution-matched sampling — *why this method*

We want the sample's per-feature distribution to match the population's. I tested
two options on this exact data before writing this cell:

- **KMeans proportional allocation** (cluster feature space, sample clusters in
  proportion to size). It matches *joint* density, but at N=50 it gave mean KS
  ≈ 0.118 — **no better than pure random** (0.118), because 50 points can't
  resolve joint structure and KS only sees marginals anyway.
- **Best-of-random (used below):** draw many random subsets, keep the one whose
  **mean per-feature KS distance to the population is smallest.** It directly
  optimizes the objective you asked for (marginal closeness) and beats random.

`ks_2samp` is the two-sample Kolmogorov–Smirnov statistic: the largest gap
between two empirical CDFs (0 = identical shape). We average it across the 10
features and minimize that. This is measured, not assumed — the verification
cell reprints the numbers so you can see the chosen sample beats a random one.

In [ ]:
def mean_ks(sample_df, population_df, features):
    # Average two-sample KS distance across features (lower = closer distributions).
    return float(np.mean([ks_2samp(sample_df[f], population_df[f]).statistic
                          for f in features]))


def representative_sample(population_df, features, n, n_draws=2000, seed=667):
    # Best-of-random: sample many candidate subsets, keep the most representative.
    rng = np.random.default_rng(seed)
    pop_cols = {f: population_df[f].to_numpy() for f in features}
    pop_sorted = {f: np.sort(v) for f, v in pop_cols.items()}

    best_ks, best_idx = np.inf, None
    for _ in range(n_draws):
        idx = rng.choice(len(population_df), size=n, replace=False)
        ks = np.mean([ks_2samp(pop_cols[f][idx], pop_sorted[f]).statistic for f in features])
        if ks < best_ks:
            best_ks, best_idx = ks, idx
    return population_df.iloc[best_idx].reset_index(drop=True), best_ks

In [ ]:
sample, sample_ks = representative_sample(pop, FEATURES, N_SAMPLES,
                                          n_draws=N_DRAWS, seed=SEED)

# Baseline: what a single plain-random draw of the same size typically scores.
rng = np.random.default_rng(SEED)
random_ks = np.mean([mean_ks(pop.sample(N_SAMPLES, random_state=int(s)), pop, FEATURES)
                     for s in rng.integers(0, 1_000_000, 20)])

print(f"Chosen sample size : {len(sample)}")
print(f"Mean KS (chosen)   : {sample_ks:.4f}   <- distribution-matched")
print(f"Mean KS (random x20): {random_ks:.4f}   <- baseline\n")

print(f"{'feature':<16}{'pop mean':>11}{'samp mean':>11}{'pop std':>10}{'samp std':>10}{'KS':>7}")
for f in FEATURES:
    print(f"{f:<16}{pop[f].mean():>11.3f}{sample[f].mean():>11.3f}"
          f"{pop[f].std():>10.3f}{sample[f].std():>10.3f}"
          f"{ks_2samp(sample[f], pop[f]).statistic:>7.3f}")

### Visual check — sample (orange) vs population (grey)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, f in zip(axes.ravel(), FEATURES):
    ax.hist(pop[f], bins=30, density=True, color="0.7", label="population")
    ax.hist(sample[f], bins=30, density=True, color="tab:orange", alpha=0.55, label="sample")
    ax.set_title(f, fontsize=9)
    ax.tick_params(labelsize=7)
axes.ravel()[0].legend(fontsize=8)
fig.suptitle("Audio-feature distributions: sample vs population", fontsize=12)
fig.tight_layout()
plt.show()

## 4. Download the audio

For each sampled track we call `spotify_dl` into its **own folder**
`data/audio/<spotify_track_id>/`. Per-track folders make it trivial to map a
file back to its id no matter how `spotify_dl` names the mp3 internally — we just
glob for the mp3 inside. Tracks already downloaded are skipped, failures are
logged and we keep going.

In [ ]:
# Credential guard — reads from the environment, never stores secrets.
missing = [v for v in ("SPOTIPY_CLIENT_ID", "SPOTIPY_CLIENT_SECRET") if not os.environ.get(v)]
if missing:
    raise EnvironmentError(
        "Missing Spotify API credentials: " + ", ".join(missing) + "\n"
        "Create a free app at https://developer.spotify.com/dashboard and, in the shell "
        "you launched Jupyter from, run:\n"
        "  export SPOTIPY_CLIENT_ID=your_client_id\n"
        "  export SPOTIPY_CLIENT_SECRET=your_client_secret\n"
        "Then restart the kernel. Do NOT paste secrets into this notebook."
    )
print("Spotify credentials found in environment.")

In [ ]:
def track_url(tid):
    return f"https://open.spotify.com/track/{tid}"

def find_mp3(tid):
    hits = sorted((AUDIO_DIR / str(tid)).glob("**/*.mp3"))
    return str(hits[0].relative_to(ROOT)) if hits else None

results = []
for i, row in sample.iterrows():
    tid = row[ID_COL]
    out_dir = AUDIO_DIR / str(tid)

    if find_mp3(tid):                       # already have it -> skip the network call
        status = "skipped_existing"
    else:
        out_dir.mkdir(parents=True, exist_ok=True)
        cmd = ["spotify_dl", "-l", track_url(tid), "-o", str(out_dir)]
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=DOWNLOAD_TIMEOUT)
            status = "ok" if find_mp3(tid) else f"failed(rc={proc.returncode})"
        except subprocess.TimeoutExpired:
            status = "timeout"
        except FileNotFoundError:
            raise RuntimeError("`spotify_dl` not found on PATH. Install it: pip install spotify_dl")
        time.sleep(SLEEP_BETWEEN)           # be polite between requests

    rec = row.to_dict()
    rec["download_status"] = status
    rec["audio_path"] = find_mp3(tid)
    results.append(rec)
    name = str(row.get("track_name", ""))[:45]
    print(f"[{i + 1:>2}/{len(sample)}] {tid}  {name!r:47}  -> {status}")

## 5. Save a manifest (id → file → features) for Stage 6

In [ ]:
manifest = pd.DataFrame(results)
manifest_path = AUDIO_DIR / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

n_ok = (manifest["download_status"].isin(["ok", "skipped_existing"])).sum()
print(f"Downloaded/most present: {n_ok}/{len(manifest)}")
print("Status counts:")
print(manifest["download_status"].value_counts().to_string())
print(f"\nManifest written to: {manifest_path.relative_to(ROOT)}")
manifest[[ID_COL, "track_name", "download_status", "audio_path"]].head(10)

## Next (Stage 6)

`manifest.csv` maps each `spotify_track_id` → local mp3 → its Spotify features
and popularity. Stage 6 loads each mp3 with `librosa`, extracts richer
descriptors (MFCCs, spectral contrast, onset density, dynamic range), and we
re-run the Model B residual test to see whether they push Spearman past the
current 0.198 — the whole reason for sourcing this audio.